In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

## Подготовка данных для обучения

In [ ]:
# **Загрузим данные для обучения**

df = pd.read_csv('E:/ML/housing-prices-ml/data/raw/train.csv')
y = np.log1p(df['SalePrice'])
X = df.drop(columns=['SalePrice', 'Id'])

# **Разделим данные на обучающую и тестовыую выборки**

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    train_size=0.8,
    shuffle=True,
    random_state=46)

## Пайплайн обучения

Разделим признаки на числовые и катгориальные для корректного построения пайплайна обучения


In [10]:
num_features = X.select_dtypes('number').columns.to_list()
cat_features = X.select_dtypes('str').columns.to_list()

Преобразования по результатам EDA

In [11]:
num_features.remove('MSSubClass')

cat_features.append('MSSubClass')

absence_features = [
    'PoolQC',
    'FireplaceQu',
    'GarageQual',
    'GarageCond',
    'GarageFinish',
    'GarageType',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'Alley',
    'Fence',
    'MiscFeature',
    'MasVnrType'
]

regular_cat_features = []

for feature in cat_features:
    if feature not in absence_features:
        regular_cat_features.append(feature)

Заполнение пропусков для baseline реализуем стандартными стратегиями: моды и медианы  
Признаки, отвечающие за отсутствие чего-либо заполняем соответствующей категорией

In [12]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

absence_pipline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

regular_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

**Базовый препроцессор**

In [13]:
preprocessor = ColumnTransformer([
    ('cat_absence', absence_pipline, absence_features),
    ('cat_reg', regular_cat_pipeline, regular_cat_features),
    ('num', num_pipeline, num_features)
])

Создадим пайплайн бэйзлайна с предобработкой и моделью - решающим деревом

In [14]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=46))
])

**Кросс валидация**

In [15]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=46
)

cv_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring='neg_root_mean_squared_error'
)

cv_rmse = -cv_scores

print(f'CV RMSE: {cv_rmse.mean():.2f} ± {cv_rmse.std():.2f}')

CV RMSE: 0.20 ± 0.02
